# 🚀 ClaimGuard AI — Production Kaggle API Server\n\n**Run ALL 4 cells in order.** After Cell 4, you'll get a PUBLIC URL.\nPaste that URL into the **Connect Node** button on your local frontend.\n\n| Cell | Function |\n|------|----------|\n| 1 | Install dependencies |\n| 2 | Install Ollama + Pull Llama3:8b |\n| 3 | Load ClaimGuard engine (OCR, AI, Blockchain) |\n| 4 | Start Flask API server with ngrok tunnel |\n\n**Requirements:** Kaggle notebook with GPU (P100) enabled.

In [ ]:
# ================================================================
# CELL 1: Install All Dependencies (Run First)
# ================================================================

!pip install -q requests pdf2image flask flask-cors pyngrok
!apt-get update -y -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
print('✅ Dependencies installed!')

In [ ]:
# ================================================================
# CELL 2: Install & Start Ollama + Pull Llama3
# ================================================================

import os
import subprocess
import time
import requests

print('🧹 Cleaning old Ollama install...')
!rm -rf /usr/local/lib/ollama
!rm -rf /usr/local/bin/ollama

print('📦 Installing dependencies...')
!apt-get install -y -qq curl zstd ca-certificates

print('⬇️ Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh

print('🚀 Starting Ollama server...')
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['OLLAMA_NUM_PARALLEL'] = '1'
env['OLLAMA_MAX_LOADED_MODELS'] = '1'

ollama_process = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(10)

def check_ollama():
    try:
        r = requests.get('http://localhost:11434/api/tags')
        return r.status_code == 200
    except:
        return False

if check_ollama():
    print('✅ Ollama is RUNNING!')
else:
    print('❌ Ollama failed to start')

print('📥 Pulling llama3:8b model...')
!ollama pull llama3:8b
print('✅ Ollama + Llama3 Ready!')

In [ ]:
# ================================================================
# CELL 3: ClaimGuard Engine (OCR, Evaluation, Simulation, Comparison)
# ================================================================

import io
import re
import json
import time
import base64
import requests
import hashlib
from datetime import datetime, timezone
from pdf2image import convert_from_path

OCR_SPACE_API_KEY = 'K87093604688957'
OLLAMA_API_URL = 'http://localhost:11434/api/generate'

# ---------- OCR TEXT EXTRACTION ----------
def extract_text_from_file(file_path):
    if not file_path: return ''
    file_path_lower = file_path.lower()
    
    if file_path_lower.endswith(('.png', '.jpg', '.jpeg')):
        print(f'Processing Image via OCR.space API...')
        with open(file_path, 'rb') as f:
            base64_encoded = base64.b64encode(f.read()).decode('utf-8')
        payload = {
            'apikey': OCR_SPACE_API_KEY,
            'base64Image': f'data:image/jpeg;base64,{base64_encoded}',
            'language': 'eng',
            'isOverlayRequired': False
        }
        try:
            response = requests.post('https://api.ocr.space/parse/image', data=payload)
            result = response.json()
            if not result.get('IsErroredOnProcessing'):
                return result['ParsedResults'][0]['ParsedText']
            return ''
        except Exception as e:
            print(f'OCR Error: {e}')
            return ''
    elif file_path_lower.endswith('.pdf'):
        print(f'Converting PDF to images for OCR...')
        try:
            images = convert_from_path(file_path, dpi=100)
            full_text = ''
            for i, img in enumerate(images):
                print(f'Processing PDF Page {i+1}...')
                img_byte_arr = io.BytesIO()
                img.save(img_byte_arr, format='JPEG', quality=40)
                base64_encoded = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')
                payload = {
                    'apikey': OCR_SPACE_API_KEY,
                    'base64Image': f'data:image/jpeg;base64,{base64_encoded}',
                    'language': 'eng',
                    'isOverlayRequired': False
                }
                try:
                    response = requests.post('https://api.ocr.space/parse/image', data=payload)
                    result = response.json()
                    if not result.get('IsErroredOnProcessing'):
                        full_text += result['ParsedResults'][0]['ParsedText'] + '\\n\\n'
                except Exception as e:
                    print(f'OCR Page {i+1} Error: {e}')
                time.sleep(1)
            return full_text
        except Exception as e:
            print(f'PDF Error: {e}')
            return ''
    else:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except:
            return 'UNSUPPORTED FILE'

# ---------- OLLAMA CALLER ----------
def call_ollama(prompt, system_prompt, json_format=True):
    payload = {
        'model': 'llama3:8b',
        'prompt': prompt,
        'system': system_prompt,
        'stream': False,
        'options': {'num_gpu': 99, 'num_predict': -1}
    }
    if json_format:
        payload['format'] = 'json'
    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=300)
        response.raise_for_status()
        return response.json().get('response', '{}')
    except Exception as e:
        print(f'Ollama error: {e}')
        return '{}'

# ---------- CLAIM EVALUATOR (v10 Logic) ----------
def car_claim_evaluator(policy_text, bill_text):
    print('🔍 Analyzing Bill against Policy...')
    sys_prompt = '''You are an Automated Car Insurance Claim Evaluation Engine.
Analyze the policy and bill documents. Determine what is covered, what is NOT covered, and the final payable amount.
Return ONLY this JSON:
{
  "total_bill_amount": number,
  "total_covered_amount": number,
  "total_not_covered_amount": number,
  "final_payable_by_insurer": number,
  "breakdown": [{"item": "...", "cost": number, "covered": true/false, "payable_amount": number, "reason": "short explanation", "category": "repair/part/labor/service"}],
  "summary": { "covered_items": [], "not_covered_items": [], "key_reasons": [] },
  "human_readable_summary": "Insurance will pay: XXXXX | You must pay: XXXXX | Main uncovered items: ..."
}
RULES: Be precise, no text outside JSON, if unclear mark NOT covered.'''
    prompt = f'POLICY:\n{policy_text}\n\nBILL:\n{bill_text}'
    raw = call_ollama(prompt, sys_prompt)
    try:
        return json.loads(raw)
    except:
        return {'error': 'JSON parse failed', 'raw': raw}

# ---------- SCENARIO SIMULATOR (v8 Logic) ----------
def parse_scenario(user_query):
    sys_prompt = 'Convert user insurance scenario to JSON with keys: treatment, hospital, urgency, input_raw, estimated_cost.'
    resp = call_ollama(user_query, system_prompt=sys_prompt)
    try:
        return json.loads(resp)
    except:
        return {'input_raw': user_query, 'hospital': 'Unknown', 'treatment': 'Unknown'}

def elite_decision_engine(scenario_json, policy_text):
    sys_prompt = '''You are an Elite AI Risk Analyst and Insurance Claim Auditor.
OUTPUT STRICTLY IN THIS JSON FORMAT:
{
  "assumptions": ["Assumption: [...] Reason: [...]"],
  "calculation_steps": ["Step 1: ...", "Step 2: ..."],
  "final_payout": "string",
  "deductions": ["deduction list"],
  "risks": ["rejection risks"],
  "hidden_insights": ["insights"],
  "claim_probability": "string",
  "risk_score": "string",
  "coverage_efficiency": "string",
  "future_prediction": "string",
  "optimization_suggestions": ["strategies"]
}'''
    prompt = f'''INPUT:\n1. Scenario:\n{json.dumps(scenario_json, indent=2)}\n\n2. Policy Text:\n{policy_text[:3000]}\n\nExecute analysis. Return ONLY JSON.'''
    try:
        resp = call_ollama(prompt, system_prompt=sys_prompt)
        return json.loads(resp)
    except Exception as e:
        return {'error': str(e)}

def run_simulation(policy_text, user_query):
    scenario = parse_scenario(user_query)
    return elite_decision_engine(scenario, policy_text)

# ---------- POLICY COMPARISON ENGINE ----------
def compare_policies(policy_texts, comparison_params=''):
    sys_prompt = '''You are an Expert Insurance Policy Comparison Engine.
OUTPUT STRICTLY IN THIS JSON FORMAT:
{
  "policies_count": number,
  "comparison_table": [{"parameter": "...", "policy_1": "...", "policy_2": "...", "winner": "Policy 1/Policy 2/Tie", "reason": "why"}],
  "overall_winner": "Policy 1" or "Policy 2",
  "overall_reason": "Detailed reason",
  "risk_analysis": {"policy_1_risks": [], "policy_2_risks": []},
  "human_readable_summary": "Policy 1 vs Policy 2: ..."
}'''
    policies_str = ''
    for i, text in enumerate(policy_texts):
        policies_str += f'\\n--- POLICY {i+1} ---\\n{text[:3000]}\\n'
    params = comparison_params if comparison_params else 'Compare all: coverage, deductibles, exclusions, premiums, limits'
    prompt = f'COMPARISON REQUEST:\nParameters: {params}\n{policies_str}\n\nReturn ONLY valid JSON.'
    raw = call_ollama(prompt, sys_prompt)
    try:
        return json.loads(raw)
    except:
        return {'error': 'Comparison parse failed', 'raw': raw}

# ---------- BLOCKCHAIN MOCK ----------
def record_blockchain(policy_name, bill_name, evaluation):
    timestamp = datetime.now(timezone.utc).isoformat()
    total_bill = evaluation.get('total_bill_amount', 0)
    record = {
        'app': 'ClaimGuardAI', 'v': '4.0', 'ts': timestamp,
        'policy': str(policy_name)[:30], 'bill': str(bill_name)[:30],
        'bill_amt': total_bill, 'payable': evaluation.get('final_payable_by_insurer', 0)
    }
    data_str = json.dumps(record, sort_keys=True)
    data_hash = hashlib.sha256(data_str.encode()).hexdigest()[:32]
    sig_bytes = hashlib.sha256(f'{data_hash}{timestamp}'.encode()).digest()[:32]
    import base58
    signature = base58.b58encode(sig_bytes).decode()
    return {
        'success': True, 'signature': signature, 'hash': data_hash,
        'explorer_url': f'https://explorer.solana.com/tx/{signature}?cluster=devnet',
        'network': 'solana-devnet'
    }

print('✅ ClaimGuard Engine loaded! All modules ready.')

In [ ]:
# ================================================================
# CELL 4: Launch Production API Server (Flask + ngrok)
# ================================================================
#
# This exposes your Kaggle GPU as a remote API that your
# local frontend connects to via the 'Connect Node' button.
#
# After running this cell, copy the PUBLIC URL and paste
# it into your frontend's 'Connect Node' dialog.
#

import os
import json
import time
import shutil
import tempfile
from datetime import datetime, timezone
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
import logging
logging.getLogger('werkzeug').setLevel(logging.ERROR)
logging.getLogger('pyngrok').setLevel(logging.ERROR)
os.environ['WERKZEUG_RUN_MAIN'] = 'true'
CORS(app)

UPLOAD_DIR = '/tmp/claimguard_uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

def save_upload(file_storage, prefix='file'):
    filename = f'{prefix}_{int(time.time())}_{file_storage.filename}'
    path = os.path.join(UPLOAD_DIR, filename)
    file_storage.save(path)
    return path

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'operational', 'service': 'ClaimGuard AI Kaggle Node', 'gpu': 'P100'})

@app.route('/api/evaluate', methods=['POST'])
def evaluate():
    start = time.time()
    policy_file = request.files.get('policy_file')
    bill_file = request.files.get('bill_file')
    if not policy_file or not bill_file:
        return jsonify({'error': 'Upload both policy_file and bill_file'}), 400
    
    policy_path = save_upload(policy_file, 'policy')
    bill_path = save_upload(bill_file, 'bill')
    
    try:
        print(f'Extracting text from {policy_file.filename}...')
        policy_text = extract_text_from_file(policy_path)
        print(f'Extracting text from {bill_file.filename}...')
        bill_text = extract_text_from_file(bill_path)
        
        if not policy_text or not bill_text:
            return jsonify({'error': 'OCR failed to extract text from one or both documents'}), 400
        
        print('Running AI evaluation...')
        evaluation = car_claim_evaluator(policy_text, bill_text)
        
        blockchain = record_blockchain(policy_file.filename, bill_file.filename, evaluation)
        
        elapsed = round(time.time() - start, 3)
        return jsonify({
            'success': True,
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'processing_time_ms': elapsed,
            'files': {'policy': policy_file.filename, 'bill': bill_file.filename},
            'evaluation': evaluation,
            'blockchain': blockchain,
            'trust': {'ai_verified': True, 'blockchain_verified': True}
        })
    finally:
        for p in [policy_path, bill_path]:
            try: os.remove(p)
            except: pass

@app.route('/api/simulate', methods=['POST'])
def simulate():
    start = time.time()
    policy_file = request.files.get('policy_file')
    user_query = request.form.get('user_query', '')
    if not policy_file:
        return jsonify({'error': 'Upload policy_file'}), 400
    
    policy_path = save_upload(policy_file, 'sim_policy')
    try:
        policy_text = extract_text_from_file(policy_path)
        if not policy_text:
            return jsonify({'error': 'OCR failed'}), 400
        result = run_simulation(policy_text, user_query)
        elapsed = round(time.time() - start, 3)
        return jsonify({'success': True, 'processing_time_ms': elapsed, 'simulation': result})
    finally:
        try: os.remove(policy_path)
        except: pass

@app.route('/api/compare', methods=['POST'])
def compare():
    start = time.time()
    files = request.files.getlist('policy_files')
    params = request.form.get('comparison_params', '')
    if len(files) < 2:
        return jsonify({'error': 'Upload at least 2 policy files'}), 400
    
    paths = []
    texts = []
    try:
        for f in files:
            path = save_upload(f, 'comp')
            paths.append(path)
            texts.append(extract_text_from_file(path))
        result = compare_policies(texts, params)
        elapsed = round(time.time() - start, 3)
        return jsonify({'success': True, 'processing_time_ms': elapsed, 'comparison': result})
    finally:
        for p in paths:
            try: os.remove(p)
            except: pass

# --- Start ngrok tunnel and Flask server ---
print('\n🌐 Starting ngrok tunnel...')
public_url = ngrok.connect(5000).public_url
print(f'''
═══════════════════════════════════════════════════════════
  🚀 CLAIMGUARD AI KAGGLE NODE IS LIVE!
═══════════════════════════════════════════════════════════
  
  📡 PUBLIC API URL:
  
     {public_url}
  
  📋 WHAT TO DO:
  1. Copy the URL above
  2. Open http://localhost:8000 in your browser
  3. Click 'Connect Node' button in the top-right
  4. Paste the URL and click OK
  5. Upload documents and run analysis!
  
  🔗 Available Endpoints:
     GET  {public_url}/api/health
     POST {public_url}/api/evaluate
     POST {public_url}/api/simulate
     POST {public_url}/api/compare

═══════════════════════════════════════════════════════════
''')

app.run(port=5000)